# 180 mm radius crossover experiment

Compare **r = 5, 7, 9 mm** with continuous motor drive through the full stroke.

- Total length **180 mm**; dead/total ratio **49.5/157.5**: dead length **56.571 mm**, stroke **123.429 mm**.
- Body mass **20–500 g**; foot mass **zero**; rotor inertia **1.86e-6 kg m²**.
- Recommended linear motor: **0.1403 Nm stall torque**, **6645 rpm free speed**. Its complete positive-torque curve lies below the measured 4 V / 120 A envelope.

The separation score is the smallest adjacent-radius percentage energy gap at 20 and 500 g, with the ranking reversing **9 > 7 > 5** to **5 > 7 > 9**. At fixed mass, energy and height have identical percentage gaps. The height constraint is **>0.2 m everywhere**; optimization uses a **0.21 m** floor.

With length and rotor inertia fixed, differential evolution searched stall torque = 0.005–0.25999 Nm and free speed = 1000–21643 rpm. Two seeds converged to **0.140254 Nm, 6644.35 rpm**, scoring **10.071%**. The rounded settings below retain **10.067%**, with minimum predicted height **0.2101 m**. This is the best numerical result found for this metric and the current rotor, not a proof of a global optimum.

The motor parameters specify the assumed linear curve; calibrate the actual curve before testing. The model uses a rigid, lossless TSA and release at the prescribed stroke endpoint.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
from scipy.optimize import brentq

L = 0.180
dead_ratio = 49.5 / 157.5
I = 1.86e-6
g = 9.81
stall_torque = 0.1403
free_rpm = 6645
radii = [0.005, 0.007, 0.009]
masses = np.linspace(0.020, 0.500, 241)


def simulate(mass, radius, rtol=1e-8, stall=stall_torque, rpm=free_rpm, inertia=I):
    free_speed = rpm * 2*np.pi/60
    theta_end = L*np.sqrt(1 - dead_ratio**2)/radius

    def dynamics(t, state):
        theta, speed = state
        root = np.sqrt(L**2 - (radius*theta)**2)
        q = radius**2*theta/root
        dq = radius**2*L**2/root**3
        torque = stall*(1 - speed/free_speed)
        accel = (torque - mass*g*q - mass*q*dq*speed**2)/(inertia + mass*q**2)
        return [speed, accel]

    def stop(t, state):
        return state[0] - theta_end

    stop.terminal = True
    stop.direction = 1
    sol = solve_ivp(dynamics, (0, 5), [0, 0], events=stop, method='DOP853',
                    max_step=0.02*theta_end/free_speed, dense_output=True,
                    rtol=rtol, atol=rtol/100)
    assert sol.success and len(sol.t_events[0]), 'Stroke not completed'
    duration = sol.t_events[0][0]
    speed = sol.y_events[0][0][1]
    q_end = radius*np.sqrt(1 - dead_ratio**2)/dead_ratio
    velocity = q_end*speed
    return dict(height_m=velocity**2/(2*g), energy_J=0.5*mass*velocity**2,
                velocity_m_s=velocity, duration_s=duration,
                rotor_energy_J=0.5*inertia*speed**2, sol=sol)


rows = []
for mass in masses:
    for radius in radii:
        result = simulate(mass, radius)
        result.pop('sol')
        rows.append(dict(mass_g=mass*1000, radius_mm=radius*1000, **result))
results = pd.DataFrame(rows)
assert results['height_m'].min() > 0.2
print(f"Lowest predicted jump over the whole sweep: {results['height_m'].min():.4f} m")


Lowest predicted jump over the whole sweep: 0.2101 m


In [2]:
def height(mass, radius):
    return simulate(mass, radius)['height_m']

crossings = []
for r1, r2 in [(0.009, 0.007), (0.007, 0.005)]:
    mass = brentq(lambda m: height(m, r1)-height(m, r2), 0.020, 0.500)
    crossings.append(dict(radii_mm=f'{r1*1000:.0f} → {r2*1000:.0f}', mass_g=mass*1000))
crossings = pd.DataFrame(crossings)
middle = brentq(lambda m: height(m, 0.005)-height(m, 0.009), 0.020, 0.500)
test_masses = [0.020, round(middle*1000)/1000, 0.500]
points = []
for mass in test_masses:
    h = np.array([height(mass, r) for r in radii])
    order = np.argsort(h)
    points.append(dict(mass_g=mass*1000, height_5_m=h[0], height_7_m=h[1], height_9_m=h[2],
                       winner_mm=radii[order[-1]]*1000,
                       winner_lead_pct=100*(h[order[-1]]/h[order[-2]]-1)))
points = pd.DataFrame(points)
low = points.iloc[0][['height_5_m', 'height_7_m', 'height_9_m']].to_numpy(dtype=float)
high = points.iloc[-1][['height_5_m', 'height_7_m', 'height_9_m']].to_numpy(dtype=float)
gaps = np.r_[low[1:]/low[:-1]-1, high[:-1]/high[1:]-1]
assert gaps.min() > 0.10
print(f'Minimum adjacent-radius gap at the two endpoints: {100*gaps.min():.3f}%')
display(crossings.round(2), points.round(4))


Minimum adjacent-radius gap at the two endpoints: 10.067%
  radii_mm  mass_g
0    9 → 7   89.94
1    7 → 5  218.63    mass_g  height_5_m  height_7_m  height_9_m  winner_mm  winner_lead_pct
0    20.0      2.4043      3.0084      3.3114        9.0          10.0728
1   152.0      0.7127      0.7371      0.7126        7.0           3.4217
2   500.0      0.2664      0.2421      0.2101        5.0          10.0671


In [3]:
energy = results.pivot(index='mass_g', columns='radius_mm', values='energy_J')
fig, ax = plt.subplots(figsize=(8, 5))
for radius, color in zip([5, 7, 9], ['#6f369b', '#168a88', '#d38900']):
    ax.plot(energy.index, energy[radius], color=color, linewidth=2, label=f'r = {radius} mm')
for mass in crossings['mass_g']:
    ax.axvline(mass, color='gray', linestyle=':', linewidth=0.9)
    ax.text(mass, 0.98, f'{mass:.0f} g', transform=ax.get_xaxis_transform(),
            ha='center', va='top', color='gray')
ax.set(title='180 mm length', xlabel='Body mass (g)', ylabel='Takeoff energy (J)', xlim=(20, 500))
ax.grid(alpha=0.25)
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig('results/radius_experiment.png', dpi=180)


## Brief experimental procedure

1. Calibrate one fixed motor curve close to **0.1403 Nm stall / 6645 rpm free speed**. Set total length to **180 mm**, dead length to **56.57 mm**, and use the same **123.43 mm stroke** for all radii.
2. Test **r = 5, 7, 9 mm** at total moving body masses **20, 50, 75, 90, 110, 150, 190, 220, 250, 350, 500 g**. Keep the rotor, string construction, and release geometry fixed; drive through stroke completion.
3. Collect five repetitions per radius/mass, randomizing radius order. Measure takeoff velocity at release and calculate **E = ½mv²**, or measure apex height above release and calculate **E = mgh**.
4. Refine mass in 5 g steps near the expected crossovers: **9 → 7 mm at 90 g**, and **7 → 5 mm at 219 g**. Plot mean energy with uncertainty bars.

The 7 mm optimum is broad: near **152 g**, it is about **3.4%** ahead. Separation is strongest at the mass extremes. The height floor is a model prediction; friction and the measured motor curve can shift both height and crossings.

The checks below verify the full target curve against the smooth measured 4 V / 120 A motor envelope, stroke completion, positive body force, solver convergence, and energy balance. Envelope fit source: `flywheeljumper/main/firmware/forcetorquemotor2/data/logs4_highspeed_defaultservo/motormodel.ipynb`.


In [4]:
free_data = [2049.7002, 4224.7279, 6390.3895, 8368.1081, 10005.7876,
             12022.9659, 14866.4361, 17137.8066, 19872.9528, 21643.4236]
voltage = 4.0
measured_free = np.interp(voltage, np.arange(1, 11), free_data)
soft_voltage = voltage - 0.683395924*np.logaddexp(0, (voltage-4.84445085)/0.683395924)
slope = -4.84787835e-5 + 5.32297125e-6*soft_voltage
speeds = np.linspace(0, free_rpm, 10001)
target = stall_torque*(1-speeds/free_rpm)
envelope = np.minimum(slope*(speeds-measured_free), 0.259990)
assert (target <= envelope).all()
for mass in test_masses:
    for radius in radii:
        a = simulate(mass, radius)
        b = simulate(mass, radius, rtol=1e-10)
        assert np.isclose(a['height_m'], b['height_m'], rtol=1e-6)
        t = np.unique(np.r_[np.linspace(0, b['duration_s'], 10001),
                            np.geomspace(1e-8, b['duration_s'], 10001)])
        theta, speed = b['sol'].sol(t)
        root = np.sqrt(L**2-(radius*theta)**2)
        q = radius**2*theta/root
        dq = radius**2*L**2/root**3
        torque = stall_torque*(1-speed/(free_rpm*2*np.pi/60))
        force = mass*(I*(g+dq*speed**2)+q*torque)/(I+mass*q**2)
        assert force.min() >= 0
        work = trapezoid(torque*speed, t)
        stored = b['energy_J']+b['rotor_energy_J']+mass*g*L*(1-dead_ratio)
        assert np.isclose(work, stored, rtol=1e-5)
print('Verified envelope compliance, stroke completion, positive body force, numerical convergence, and energy balance.')


Verified envelope compliance, stroke completion, positive body force, numerical convergence, and energy balance.


In [5]:
fig_motor, ax_motor = plt.subplots(figsize=(8, 5))
for volts, style in [(10, '-'), (4, '--')]:
    free = np.interp(volts, np.arange(1, 11), free_data)
    rpm_grid = np.linspace(0, free, 1000)
    soft = volts-0.683395924*np.logaddexp(0, (volts-4.84445085)/0.683395924)
    fitted_slope = -4.84787835e-5+5.32297125e-6*soft
    available = np.minimum(fitted_slope*(rpm_grid-free), 0.259990)
    ax_motor.plot(rpm_grid, available, style, color='#42799a', linewidth=1.8,
                  label=f'Fitted {volts} V envelope')
    if volts == 10:
        ax_motor.fill_between(rpm_grid, 0, available, color='#42799a', alpha=0.12)
selected_rpm = np.linspace(0, free_rpm, 1000)
selected_torque = stall_torque*(1-selected_rpm/free_rpm)
ax_motor.plot(selected_rpm, selected_torque, color='#c65c16', linewidth=2.8,
              label=f'Selected: {stall_torque:.4f} Nm, {free_rpm:.0f} rpm')
ax_motor.plot([0, free_rpm], [stall_torque, 0], 'o', color='#c65c16', markersize=5)
ax_motor.set(title='Motor torque–speed envelope, 120 A limit', xlabel='Speed (rpm)',
             ylabel='Shaft torque (Nm)', xlim=(0, 22500), ylim=(0, 0.28))
ax_motor.grid(alpha=0.25)
ax_motor.legend()
fig_motor.tight_layout()



## Comparison within the full 10 V envelope

The original search already covered the complete 10 V capability. The current curve maximizes the weakest **percentage** gap. Optimizing the weakest **absolute energy gap in joules** instead gives about **0.25999 Nm stall / 8233 rpm free speed**, rounded here to **8230 rpm**. Two differential-evolution seeds gave the same optimum, with the same full ranking reversal and 0.21 m height floor.

The alternative roughly doubles the weakest energy gap at the mass endpoints (**0.059 → 0.117 J**) and gives taller jumps (**minimum 0.457 m**), while reducing the weakest percentage gap (**10.1 → 4.8%**). Its crossovers are around **126 g** and **308 g**. Both curves remain driven throughout the full stroke.

The strongest admissible linear motor, 0.25999 Nm / 21643 rpm, instead makes 5 mm optimal above about 22 g and eliminates the 9 mm winning region over 20–500 g. Maximizing motor capability alone therefore does not maximize crossover separation. The comparison below leaves the original recommendation intact.

In [6]:
alternative_stall, alternative_rpm = 0.25999, 8230
rows = []
for mass in masses:
    for radius in radii:
        result = simulate(mass, radius, stall=alternative_stall, rpm=alternative_rpm)
        result.pop('sol')
        rows.append(dict(mass_g=mass*1000, radius_mm=radius*1000, **result))
alternative = pd.DataFrame(rows)
assert alternative['height_m'].min() > 0.2

comparison = []
fig_compare, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, name, data, stall, rpm in zip(axes,
        ['Current: percentage gaps', 'Alternative: energy gaps'],
        [results, alternative], [stall_torque, alternative_stall], [free_rpm, alternative_rpm]):
    energy = data.pivot(index='mass_g', columns='radius_mm', values='energy_J')
    for radius, color in zip([5, 7, 9], ['#6f369b', '#168a88', '#d38900']):
        ax.plot(energy.index, energy[radius], color=color, linewidth=2, label=f'r = {radius} mm')
    for r1, r2 in [(0.009, 0.007), (0.007, 0.005)]:
        cross = brentq(lambda m: simulate(m, r1, stall=stall, rpm=rpm)['energy_J']
                       - simulate(m, r2, stall=stall, rpm=rpm)['energy_J'], 0.020, 0.500)*1000
        ax.axvline(cross, color='gray', linestyle=':', linewidth=0.9)
        ax.text(cross, 0.98, f'{cross:.0f} g', transform=ax.get_xaxis_transform(),
                ha='center', va='top', color='gray')
    low, high = energy.iloc[0].to_numpy(), energy.iloc[-1].to_numpy()
    absolute = np.r_[np.diff(low), -np.diff(high)]
    relative = np.r_[low[1:]/low[:-1]-1, high[:-1]/high[1:]-1]
    comparison.append(dict(curve=name, stall_Nm=stall, free_rpm=rpm,
                           min_gap_J=absolute.min(), min_gap_pct=100*relative.min(),
                           min_height_m=data['height_m'].min()))
    ax.set(title=name, xlabel='Body mass (g)', ylabel='Takeoff energy (J)', xlim=(20, 500), ylim=(0.4, 2.75))
    ax.grid(alpha=0.25)
    ax.legend(loc='lower right')
fig_compare.tight_layout()
fig_compare.savefig('results/radius_energy_comparison.png', dpi=180)
display(pd.DataFrame(comparison).round(4))

rpm_alt = np.linspace(0, alternative_rpm, 10001)
torque_alt = alternative_stall*(1-rpm_alt/alternative_rpm)
soft10 = 10-0.683395924*np.logaddexp(0, (10-4.84445085)/0.683395924)
slope10 = -4.84787835e-5+5.32297125e-6*soft10
assert np.all(torque_alt <= np.minimum(slope10*(rpm_alt-free_data[-1]), 0.259990))
ax_motor.plot(rpm_alt, torque_alt, color='#167564', linewidth=2.8,
              label='Energy-gap curve: 0.260 Nm, 8230 rpm')
ax_motor.legend(fontsize=9)
fig_motor.savefig('results/radius_motor_comparison.png', dpi=180)
plt.close(fig_compare)
plt.close(fig_motor)
display(fig_compare, fig_motor)


                      curve  stall_Nm  ...  min_gap_pct  min_height_m
0  Current: percentage gaps    0.1403  ...      10.0671        0.2101
1  Alternative: energy gaps    0.2600  ...       4.7538        0.4573

[2 rows x 6 columns]
Figure(1200x480) Figure(800x500)


In [7]:
results.to_csv('results/mass_sweep.csv', index=False)
alternative.to_csv('results/energy_gap_sweep.csv', index=False)
pd.DataFrame(comparison).to_csv('results/motor_comparison.csv', index=False)
